# DiscountMate Product Pricing -- Woolworths
### Author: Rebekah De Losa
### Unit: SIT782

**Purpose**

This notebook injests .csv file(s) from the Woolworths web scraper in use for T1 2026. Data are then cleaned according to DiscountMate schema and new variables are created as required. Each week of cleaned pricing data (one .csv output file) is then joined with a corresponding DiscountMate `Product ID`. To forge this join, GTIN (Barcode) is used as a surrogate Foreign Key, as it is a universal unique identifier of a product. Products are pulled from the MongoDB Database and joined with the new pricings data. After final checks are performed, the new pricings data is uploaded one document at a time to the MongoDB `product_pricings` table.  

Please note: The MongoDB `product_pricings` table is the only MongoDB table updated with new information. The `products` MongoDB table contains detailed information about ~25K Coles products obtained from the Master Coles Scrape. Pricing data from *all* retailers is joined to this master set of products to serve the front end of DiscountMate. Products from individual retailers are not uploaded to MongoDB. 

Warning: A direct connection to the MongoDB DiscountMate DB is set up in this notebook.  

**Context**

In T1 2026 the Data Engineering team redeveloped the DiscountMate ETL pipeline. This notebook was developed to ensure that the DiscountMate website was populated with products and prices during this transition period and also ensure that all teams had access to data for analysis. It has been used routinely throughout T1 2026 to upload new Woolworths product prices.

**Setup**
* Ensure a *.env* file is stored locally and contains the MongoDB connection string
* *Cell 1*: Specify the path for local storage of Scraper .csv files.
* *Cell 7*: This cell has been intentially commented-out to prevent accidental upload to the MongoDB Database

**Further information**

For further information on the DiscountMate schema used for this notebook, how to create and connect with MongoDB or get a more general overview in how it was used for DiscountMate, please see the MongoDB SOP on DiscountMate Github here: **DE/MongoDB - Legacy/MongoDB for DiscountMate - T1 2026.pdf**

### Import and setup

In [ ]:
# Libraries
import pandas as pd
from pandas.errors import EmptyDataError
import numpy as np
import os
from pymongo import MongoClient
from dotenv import load_dotenv
from bson.objectid import ObjectId

# Data stored locally
path = ""                                       # Insert folder path here (i.e. r"C:\Downloads\Woolworths"). Should point to where your .csv files are stored.
path_upld = ""                                  # Insert folder path here (i.e. r"C:\Users\mylogin\Deakin2026\Team Project B\Uploaded_Files"). Should point to where you want to store the .csv files after upload (if required).

### Connect to MongoDB and retrieve latest dataset

In [ ]:
load_dotenv()
mongo_uri = os.getenv('MONGO_URI')                       # Relies on connection string in .env

# Connect to MongoDB
client = MongoClient(mongo_uri)
database = 'DiscountMate_DB'                            # This is the T1 2026 database name

# Load database
db = client[database]

# Define functions
def get_data(table, database):
    """
    Args:
        table (str): The name of the table to retrieve from the database.
        database: The MongoDB database object to query.
    Returns:
        pd.DataFrame: A pandas dataframe of the specified table from the database.
    """
    return pd.DataFrame(database[table].find({}))

def create_data(db, tablename, newDocuments):
    """
    Args:
        db: The MongoDB database object to query.
        tablename (str): The name of the collection to insert documents into.
        newDocuments (list): A list of dictionaries, each dictionary is a new document for the database. Keys of the dictionary should correspond to the schema.

    Returns:
        tuple: A tuple containing the updated pandas dataframe and the count of documents uploaded.
    """
    docCount = 0
    collection = db[tablename]
    for doc in newDocuments:
        collection.insert_one(doc)
        docCount += 1
        print(docCount)
    table = pd.DataFrame(collection.find({}))           # re-import the table with updated documents
    return table, docCount                              # returns df of table and count of docs uploaded


# DATA FOR JOINING
prods = get_data('products', db)                        # For product_id, product_code join
prices = get_data('product_pricings', db)               # For best_price, best_unit_price -- will be made obsolete

### Establish variables to target

In [ ]:
targetvars = ['product_id',     # Mongo: Product Table
              'product_code',   # Mongo: Product Table
              'store_chain',    # Mongo: Store Table
              'date',           # Date collected: name: 'Timestamp', type: 'd/m/Y'
              'price',          # Pricing on date
              'best_price',     # Best price seen at store (historically) -- Discussion regarding whether this is required
              'best_unit_price',# Best unit price seen at store (historically) -- Discussion regarding whether this is required
              'unit_price',     # Unit price on date
              'is_on_special',  # Is on special
              'created_at']     # Date uploaded to mongo -- added at doc upload stage

varsfromfile = [var for var in targetvars if var not in ['best_price', 'best_unit_price', 'created_at']]

### Define processing functions for each variable

In [ ]:
def get_date(data, coltarget):
    # Convert to datetime
    data['date'] = pd.to_datetime(data[coltarget], unit='s')

def get_price(data, coltarget):
    # Convert to float
    data['price'] = data[coltarget].astype('float')


def get_unit_price(data, coltarget): 
    # Split size into measurement and units with regex
    data[['unit_per_prod', 'measurement']] = data[coltarget].str.extract(r"(\d+)\s*([a-zA-Z]+)")
    # Tidy up units
    # -- Lowercase
    data['measurement'] = data['measurement'].str.lower()
    # -- Dict replace to most common format
    size_dict = {'gram': 'g',
                'litre': 'l',
                'metre': 'm',
                'metres': 'm',
                'ea': 'each',
                'meters': 'm'}

    data['measurement'] = data['measurement'].replace(size_dict)
    # Unit price - put it all together
    data['unit_value'] = round(data['price'] / data['unit_per_prod'].astype(float), 3)
    data['unit_price'] = data['unit_value'].astype(str) + " per " + data['measurement']

def transform_barcode(data, coltarget):
    # Convert empty to 0 > int > str > replace any 0 back to empty
    data['gtin'] = data[coltarget].fillna(0).astype('float').astype('int').astype('str').replace({"0":None})
    data.dropna(subset=['gtin'],inplace=True) # No missing barcodes

def get_is_on_special(data, coltarget):
    data['is_on_special'] = data[coltarget].str.strip().str.lower()
    data['is_on_special'] = data['is_on_special'].map({'true': True, 'false': False})
    data['is_on_special'] = data['is_on_special'].astype(bool)

def join_prods(data, product_data):    
    # Join with dataframe to get product_id and product_code for each record with a gtin
    return pd.merge(product_data[['_id','product_code','gtin']].copy(),
                        data,
                        how='inner', # Keep only crossover data
                        on='gtin').rename(columns={'_id': 'product_id'})


### Import raw data and transform
Go to local directory and retrieve all files to be imported.
Expects .csv files are stored within 'woolworths' folder. 

In [ ]:

df_list = {}
store = 'woolworths'
barcode_list = []

for file in os.listdir(os.path.join(path, store)):
    try:
        df = pd.read_csv(os.path.join(path, store, file), dtype='str')
        df_list[file] = df
        print(f"File {file} added to dataframe dict...")
    except EmptyDataError:
        print(f"File {file} is empty. Skipping...")

# Create empty df for concatenating transformed data into
complete_df = pd.DataFrame(columns=varsfromfile + ['gtin']) # Start with all variables from file + gtin for joining. Will add best_price, best_unit_price, created_at later.

# Transform dataframes and concatenate into complete_df for upload
for namedf, df in df_list.items():
    barcode_list.append(df[['Barcode','Stockcode']].copy()) #Pair barcode with stockcode for matching
    print(f"File {namedf} has {df.shape[0]} rows")
    get_date(df,'Timestamp')                                # date
    get_price(df,'Price')                                   # price
    get_unit_price(df, 'PackageSize')                       # unit_price
    transform_barcode(df, 'Barcode')                        # gtin (for join)
    get_is_on_special(df, 'IsOnSpecial')                    # is_on_special
    joindf = join_prods(df, prods)                          # product_id, product_code
    joindf['store_chain'] = 'woolworths_generic'            # ID for the store_schain
    joindf = joindf[varsfromfile + ['gtin']].copy()

    #  Only for first upload:
    print(f"{namedf} has upload length {len(joindf)}")
    joindf.drop_duplicates(inplace=True)
    print(f"{namedf} has upload length {len(joindf)} after dropping whole row duplicates")
    joindf.drop_duplicates(subset=['gtin'], inplace=True)
    print(f"{namedf} has upload length {len(joindf)} after dropping gtin duplicates")
    complete_df = pd.concat([complete_df, joindf], ignore_index=True)

# Ensure dtypes preserved
complete_df = complete_df.astype({
    'product_id': 'object',
    'product_code': 'string',
    'store_chain': 'string',
    'price': 'float64',
    'unit_price': 'string',
    'is_on_special': 'bool',
    'gtin': 'object'
})

### Prep for Import

In [ ]:

import_data = complete_df.copy() 

print(f"Number of rows to import after dropping test records is {import_data.shape[0]}")

# Remove gtin as it is not required once join is made
import_data.drop(columns=['gtin'], inplace=True)

# Get existing pricing data in database to calculate best_price and best_unit_price
current_prices = prices[prices['store_chain'] == 'woolworths_generic'].copy()
current_prices_idx = current_prices.groupby('product_id')['price'].idxmin()
best_prices = current_prices.loc[current_prices_idx, ['product_id', 'price', 'unit_price']].copy()
best_prices.rename(columns={'price':'best_price', 'unit_price':'best_unit_price'}, inplace=True)

if best_prices.shape[0] != best_prices['product_id'].nunique():
    raise ValueError(f"More than one best price found for a product! Review.")

# Join best price data with new data for import
import_data = import_data.merge(best_prices, on='product_id', how='left')

mask = import_data['price'] < import_data['best_price']

import_data.loc[mask, 'best_price'] = import_data.loc[mask, 'price']
import_data.loc[mask, 'best_unit_price'] = import_data.loc[mask, 'unit_price']

mask = import_data['best_price'].isna()

import_data.loc[mask, 'best_price'] = import_data.loc[mask, 'price']
import_data.loc[mask, 'best_unit_price'] = import_data.loc[mask, 'unit_price']

# Check for missing price data
print(f"Total rows before drop: {import_data.shape[0]}")
print(f"Rows with missing price data: {import_data[import_data['price'].isna()].shape[0]}")
print(f"Rows with complete price data: {import_data[~import_data['price'].isna()].shape[0]}")
import_data = import_data[~import_data['price'].isna()]
print(f"Total rows after drop: {import_data.shape[0]}")

# Add created at timestamp
import_data['created_at'] = pd.to_datetime('now')
# Rearrange columns to match targetvars
import_data = import_data[targetvars]
# -- null replacement for mongo
import_data = import_data.replace({np.nan: None})
# Export to file
dte = pd.to_datetime('now').strftime('%Y%m%d')
import_data[targetvars].to_csv(path_upld + rf"\woolworths_upload_{dte}.csv", index=False)

# -- to dict
import_dict = import_data.to_dict(orient='records')
print(f"Import dict has {len(import_dict)} records")


# Upload to Mongo
When tested and verified, run cell below to upload pricing information to MongoDB.

In [ ]:
# updatedPriceTable, priceCount = create_data(db, 'product_pricings', import_dict)
# print(f"Upload finished. {priceCount} product_pricings uploaded.")